# NLP Project – Phase 1: corpus, preprocessing and application area

**Group 5 – Italian** · model: `it_core_news_md`

Corpus: 20 Italian cities – the opening paragraphs of their Italian Wikipedia articles
(248–300 words and 4–8 paragraphs per text), licence CC BY-SA 4.0.
The texts were collected with `build_corpus.py` (Wikipedia API, no scraping);
the metadata is in `sources.csv`.

## 1. Setup – Anaconda environment, TensorFlow, Keras, spaCy

Environment created with Anaconda (Section 1.3 of the tutorial):

```
conda create -n nlp-tutorial1 -c conda-forge python=3.12
conda activate nlp-tutorial1
conda install -c conda-forge spacy jupyter
pip install tensorflow keras          # conda-forge has no recent TensorFlow for Windows
python -m spacy download it_core_news_md
```

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # hide TensorFlow info messages
import tensorflow as tf
import keras
print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)

TensorFlow: 2.21.0
Keras     : 3.15.1


In [2]:
import sys
import spacy
from spacy import displacy

print('Python:', sys.version.split()[0])
print('spaCy :', spacy.__version__)

Python: 3.12.16
spaCy : 3.8.16


In [3]:
!"{sys.executable}" -m spacy info


============================== Info about spaCy ==============================

spaCy version    3.8.16                        
Location         C:\Users\laura\anaconda3\envs\nlp-tutorial1\Lib\site-packages\spacy
Platform         Windows-11-10.0.26300-SP0     
Python version   3.12.16                       
Pipelines        it_core_news_md (3.8.0)       



## 2. Loading the Italian model and processing one sentence

Installed with `python -m spacy download it_core_news_md`.

In [4]:
nlp = spacy.load('it_core_news_md')
print('Pipeline:', nlp.pipe_names)

doc = nlp("Leonardo da Vinci nacque a Vinci nel 1452 e lavorò a Firenze e a Milano.")

print(f"{'TOKEN':<10}{'LEMMA':<10}{'POS':<7}{'DEP':<12}{'HEAD':<10}ENT")
for t in doc:
    print(f"{t.text:<10}{t.lemma_:<10}{t.pos_:<7}{t.dep_:<12}{t.head.text:<10}{t.ent_type_}")

print('\nEntities:', [(e.text, e.label_) for e in doc.ents])

Pipeline: ['tok2vec', 'morphologizer', 'tagger', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']
TOKEN     LEMMA     POS    DEP         HEAD      ENT
Leonardo  Leonardo  PROPN  nsubj       nacque    PER
da        da        ADP    case        Vinci     PER
Vinci     Vinci     PROPN  nmod        Leonardo  PER
nacque    nascere   VERB   ROOT        nacque    
a         a         ADP    case        Vinci     
Vinci     Vinci     PROPN  obl         nacque    LOC
nel       in il     ADP    case        1452      
1452      1452      NUM    obl         nacque    
e         e         CCONJ  cc          lavorò    
lavorò    lavorare  VERB   conj        nacque    
a         a         ADP    case        Firenze   
Firenze   Firenze   PROPN  obl         lavorò    LOC
e         e         CCONJ  cc          Milano    
a         a         ADP    case        Milano    
Milano    Milano    PROPN  conj        Firenze   LOC
.         .         PUNCT  punct       nacque    

Entities: [('Leonardo da Vinc

## 3. Visualization with displaCy

`displacy.serve(doc, style='dep')` starts a web server on port 5000 (use `port=5001`
if the port is busy). Inside a notebook we use `displacy.render()` instead, which shows
the result directly in the cell. We also save every picture to the `images/` folder
(SVG for dependencies, HTML for entities).

In [5]:
from pathlib import Path

IMG = Path('images')
IMG.mkdir(exist_ok=True)
DEP_OPTS = {'compact': True, 'distance': 110}

def show_and_save(doc, name):
    displacy.render(doc, style='dep', jupyter=True, options=DEP_OPTS)
    displacy.render(doc, style='ent', jupyter=True)
    (IMG / f'{name}_dep.svg').write_text(
        displacy.render(doc, style='dep', options=DEP_OPTS, jupyter=False), encoding='utf-8')
    (IMG / f'{name}_ent.html').write_text(
        displacy.render(doc, style='ent', page=True, jupyter=False), encoding='utf-8')

show_and_save(doc, 'example')

Practice with a few more sentences:

In [6]:
practice = [
    "Bill Gates è il fondatore di Microsoft.",
    "Il Papa vive nella Città del Vaticano, a Roma.",
    "Ieri Maria ha comprato un gelato all'Università di Bologna.",
]
for i, s in enumerate(practice, 1):
    d = nlp(s)
    print(s, '->', [(e.text, e.label_) for e in d.ents])
    displacy.render(d, style='ent', jupyter=True)

Bill Gates è il fondatore di Microsoft. -> [('Bill Gates', 'PER'), ('Microsoft', 'ORG')]


Il Papa vive nella Città del Vaticano, a Roma. -> [('Papa', 'PER'), ('Città del Vaticano', 'LOC'), ('Roma', 'LOC')]


Ieri Maria ha comprato un gelato all'Università di Bologna. -> [('Ieri Maria', 'MISC'), ('Università di Bologna', 'LOC')]


## 4. Building the corpus

* **Topic:** Italian cities – Roma, Milano, Napoli, Torino, Palermo, Genova, Bologna, Firenze, Bari,
  Catania, Venezia, Verona, Messina, Padova, Trieste, Brescia, Parma, Taranto, Modena, Reggio Calabria.
* **Text type:** the opening paragraphs of each city's Italian Wikipedia article: whole paragraphs,
  248–300 words per text (the last paragraph is cut at the end of a sentence).
* **Source / licence:** it.wikipedia.org, CC BY-SA 4.0 (reuse allowed with attribution).
* **Files:** `corpus/text01.txt` … `corpus/text20.txt`, UTF-8, one paragraph per line.
* **Metadata:** `sources.csv` (file, title, URL, date accessed, licence, words, paragraphs).

### 4.1 Encoding check (UTF-8, no mojibake)

In [7]:
import re, csv

files = sorted(Path('corpus').glob('*.txt'))
accents = 'àèéìòùÀÈÉÌÒÙ'
for f in files:
    raw = f.read_bytes()
    raw.decode('utf-8')                       # raises an error if a file is not UTF-8
    text = raw.decode('utf-8')
    mojibake = re.findall(r'Ã.|Â.|â€', text)  # typical signs of a wrong encoding
    found = sorted({c for c in text if c in accents})
    print(f"{f.name}: {len(text.split()):>3} words, accents {''.join(found):<8} mojibake: {len(mojibake)}")

text01.txt: 272 words, accents Èàèéòù   mojibake: 0
text02.txt: 291 words, accents àèìòù    mojibake: 0
text03.txt: 292 words, accents Èàèéòù   mojibake: 0
text04.txt: 272 words, accents àèé      mojibake: 0
text05.txt: 278 words, accents àèéòù    mojibake: 0
text06.txt: 270 words, accents Èàèéìù   mojibake: 0
text07.txt: 271 words, accents Èàèòù    mojibake: 0
text08.txt: 248 words, accents Èàèéù    mojibake: 0
text09.txt: 284 words, accents Èàèìù    mojibake: 0
text10.txt: 284 words, accents Èàèù     mojibake: 0
text11.txt: 297 words, accents àèù      mojibake: 0
text12.txt: 268 words, accents àèò      mojibake: 0
text13.txt: 288 words, accents àèòù     mojibake: 0
text14.txt: 286 words, accents àèòù     mojibake: 0
text15.txt: 271 words, accents àèù      mojibake: 0
text16.txt: 286 words, accents Èàèìòù   mojibake: 0
text17.txt: 282 words, accents Èàèé     mojibake: 0
text18.txt: 259 words, accents àèéù     mojibake: 0
text19.txt: 300 words, accents àèò      mojibake: 0
text20.txt: 

### 4.2 Loading and measuring the corpus (Section 6.4)

In [8]:
from collections import Counter

texts = [f.read_text(encoding='utf-8') for f in files]

tokens = []
docs = []
for text in texts:
    d = nlp(text)
    docs.append(d)
    tokens += [t.text.lower() for t in d if t.is_alpha]

print('Texts:', len(texts))
print('Tokens:', len(tokens))
print('Unique words:', len(set(tokens)))
print(Counter(tokens).most_common(10))

Texts: 20
Tokens: 5389
Unique words: 1646
[('e', 211), ('di', 184), ('la', 142), ('il', 138), ('del', 117), ('è', 113), ('della', 97), ('città', 94), ('in', 66), ('nel', 61)]


In [9]:
# The same ranking without stop words and with lemmas, to see the topic of the corpus
content = [t.lemma_.lower() for d in docs for t in d if t.is_alpha and not t.is_stop]
print('Top 15 content lemmas:', Counter(content).most_common(15))
print('Sentences:', sum(len(list(d.sents)) for d in docs))
print('Paragraphs:', sum(len([p for p in t.splitlines() if p.strip()]) for t in texts))
print('Average tokens per text:', round(len(tokens) / len(texts), 1))

Top 15 content lemmas: [('comune', 49), ('centro', 45), ('italiano', 42), ('secolo', 33), ('il', 33), ('italia', 30), ('capitale', 29), ('omonimo', 26), ('capoluogo', 25), ('abitante', 23), ('metropolitano', 21), ('regione', 21), ('sede', 19), ('patrimonio', 19), ('territorio', 19)]
Sentences: 237
Paragraphs: 101
Average tokens per text: 269.4


**What are the ten most frequent words?** *e, di, la, il, del, è, della, città, in, nel*.
Nine of them are stop words (conjunctions, prepositions, articles and the verb *essere*); they
only show that the text is Italian. Note how many are articulated prepositions
(*del*, *della*, *nel* = preposition + article). The one exception is ***città*** (city, 94 times),
which already gives away the topic. After removing stop words the topic is clear:
*comune, centro, italiano, secolo, capitale, capoluogo, abitante, metropolitano, regione* –
the vocabulary of describing a city. (The lemma *il* still appears because the elided article
*l'* is not marked as a stop word.)

## 5. Looking for quality problems (Section 6.5)

In [10]:
problems = {}
def note(name, msg):
    problems.setdefault(name, []).append(msg)

for f, text in zip(files, texts):
    if re.search(r'<[^>]+>|&[a-z]+;|\[\d+\]', text):
        note(f.name, 'HTML tag / entity / reference mark')
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if len(lines) != len(set(lines)):
        note(f.name, 'repeated line')
    if re.search(r'\b(Modifica|Indice|Vai a|Menu|Leggi)\b', text):
        note(f.name, 'possible menu/boilerplate words')
    if re.search(r'\d \d{3}', text):
        note(f.name, 'number written with spaces (e.g. 2 743 357)')
    if re.search(r'\s-\.|\(\s*\)|\s,', text):
        note(f.name, 'leftover of a removed template (e.g. "Gamma -." or "( )")')

# sentences in another language: share of Italian stop words per sentence
for f, d in zip(files, docs):
    for s in d.sents:
        words = [t for t in s if t.is_alpha]
        if len(words) >= 5 and sum(t.is_stop for t in words) / len(words) < 0.15:
            note(f.name, f'sentence with few Italian stop words: "{s.text[:60]}..."')

# foreign words recognised by the model as MISC entities (Latin, English ...)
for f, d in zip(files, docs):
    misc = [e.text for e in d.ents if e.label_ == 'MISC']
    if misc:
        note(f.name, f'MISC entities: {misc[:4]}')

for name in sorted(problems):
    print(name)
    for m in problems[name]:
        print('   -', m)

text01.txt
   - number written with spaces (e.g. 2 743 357)
   - MISC entities: ['Cuore della cristianità cattolica']
text02.txt
   - number written with spaces (e.g. 2 743 357)
   - MISC entities: ['Mediolanum', 'editto di', 'Dal XII', 'Restaurazione']
text03.txt
   - number written with spaces (e.g. 2 743 357)
   - MISC entities: ['Cumani', 'I secolo d.C.', "Unità d'Italia", 'Umanesimo']
text04.txt
   - number written with spaces (e.g. 2 743 357)
   - leftover of a removed template (e.g. "Gamma -." or "( )")
   - MISC entities: ['XX Giochi olimpici invernali', 'ATP Finals', 'Eurovision Song Contest', 'Capitale mondiale del libro']
text05.txt
   - number written with spaces (e.g. 2 743 357)
   - MISC entities: ['Barocco', 'Réseau Art Nouveau Network']
text06.txt
   - number written with spaces (e.g. 2 743 357)
   - MISC entities: ['Superba']
text07.txt
   - number written with spaces (e.g. 2 743 357)
   - sentence with few Italian stop words: "I primi insediamenti nell'area bolognese 

In [11]:
# Italian Wikipedia writes thousands with a space, so one number becomes several tokens
d = nlp(texts[0].split('.')[0])
print([t.text for t in d][:12])

['Roma', 'è', 'un', 'comune', 'italiano', 'di', '2', '743', '357', 'abitanti', ',', 'capoluogo']


**Findings**

* No HTML tags, menus or repeated lines – the Wikipedia API returns plain text, so the corpus is clean.
* Numbers are written with spaces as thousand separators (*2 743 357 abitanti*) and are split into
  three `NUM` tokens. This changes the dependency tree (see sentence 1 below).
* One small template leftover in `text04.txt` (Torino: *nella categoria Gamma -.*), where a
  ranking symbol was lost when converting to plain text.
* Pronunciation brackets in IPA and dialect names after the city name, e.g.
  *Brescia (afi: /ˈbr(ɛ)ʃːa/, … Brèsa o Bressa in dialetto bresciano …)*. Our collection script
  removes this first bracket (with nested brackets), because it is not running Italian text.
* The sentence flagged as "few Italian stop words" (text07, Bologna) is Italian – a false
  positive of our simple heuristic (it is full of numbers and names).
* Some entities and words are not Italian or are historical names (*Partenope*, *Cumani*,
  *Mediolanum*), and the model labels many of them **MISC**.
* All 20 texts begin with the same pattern (*X è un comune italiano di N abitanti, capoluogo …*),
  which inflates words such as *comune*, *capoluogo*, *abitanti*, *omonimo*.

### 5.1 Three sentences from the corpus – dependencies and entities

In [12]:
def find_sentence(doc, start):
    return next(s.text.strip() for s in doc.sents if s.text.strip().startswith(start))

three = [
    find_sentence(docs[0], 'Roma è un comune'),   # Roma (text01)
    find_sentence(docs[2], 'Napoli nasce'),       # Napoli (text03)
    find_sentence(docs[11], 'Di origine'),        # Verona (text12)
]
for i, s in enumerate(three, 1):
    d = nlp(s)
    print(f'--- Sentence {i}:', s)
    print('Entities:', [(e.text, e.label_) for e in d.ents])
    print(' '.join(f'{t.text}/{t.pos_}' for t in d))
    show_and_save(d, f'sentence{i}')

--- Sentence 1: Roma è un comune italiano di 2 743 357 abitanti, capoluogo dell'omonima città metropolitana e della regione Lazio, nonché capitale della Repubblica Italiana.
Entities: [('Roma', 'LOC'), ('Lazio', 'LOC'), ('Repubblica Italiana', 'LOC')]
Roma/PROPN è/AUX un/DET comune/ADJ italiano/ADJ di/ADP 2/NUM 743/NUM 357/NUM abitanti/NOUN ,/PUNCT capoluogo/NOUN dell'/ADP omonima/ADJ città/NOUN metropolitana/ADJ e/CCONJ della/ADP regione/NOUN Lazio/PROPN ,/PUNCT nonché/CCONJ capitale/NOUN della/ADP Repubblica/NOUN Italiana/ADJ ./PUNCT


--- Sentence 2: Napoli nasce come Partenope, fondata dai Cumani su Pizzofalcone a fine VIII secolo a.C. su base archeologica, sebbene sporadici reperti suggeriscano una datazione più antica.
Entities: [('Napoli', 'LOC'), ('Partenope', 'LOC'), ('Cumani', 'MISC'), ('Pizzofalcone', 'LOC')]
Napoli/PROPN nasce/VERB come/ADP Partenope/PROPN ,/PUNCT fondata/VERB dai/ADP Cumani/PROPN su/ADP Pizzofalcone/PROPN a/ADP fine/NOUN VIII/ADJ secolo/NOUN a.C./ADJ su/ADP base/NOUN archeologica/ADJ ,/PUNCT sebbene/CCONJ sporadici/ADJ reperti/NOUN suggeriscano/VERB una/DET datazione/NOUN più/ADV antica/ADJ ./PUNCT


--- Sentence 3: Di origine preistorica, l'abitato venne rifondato dai Romani all'interno dell'ansa del fiume intorno alla metà del I secolo a.C., rimanendo sotto il governo dell'Impero fino al V secolo, quando venne occupato dal re germanico Teodorico il Grande.
Entities: [('Romani', 'LOC'), ('I secolo a.C.', 'MISC'), ('Impero', 'LOC'), ('Teodorico il Grande', 'PER')]
Di/ADP origine/NOUN preistorica/ADJ ,/PUNCT l'/DET abitato/NOUN venne/AUX rifondato/VERB dai/ADP Romani/PROPN all'/ADP interno/NOUN dell'/ADP ansa/NOUN del/ADP fiume/NOUN intorno/ADP alla/DET metà/NOUN del/ADP I/ADJ secolo/NOUN a.C./ADJ ,/PUNCT rimanendo/VERB sotto/ADP il/DET governo/NOUN dell'/ADP Impero/NOUN fino/ADV al/DET V/ADJ secolo/NOUN ,/PUNCT quando/SCONJ venne/AUX occupato/VERB dal/ADP re/NOUN germanico/ADJ Teodorico/PROPN il/DET Grande/PROPN ./PUNCT


In [13]:
# The details we discuss below
d1, d2, d3 = (nlp(s) for s in three)
print('S1:', [(t.text, t.pos_, t.dep_, t.head.text) for t in d1 if t.text in ('comune', '2', '743', '357', 'abitanti')])
print('S2:', [(t.text, t.lemma_, t.dep_) for t in d2 if t.text in ('fondata', 'secolo', 'suggeriscano')], [(e.text, e.label_) for e in d2.ents])
print('S3:', [(t.text, t.pos_, t.dep_) for t in d3 if t.text in ('abitato', 'venne', 'rifondato', 'Grande')], [(e.text, e.label_) for e in d3.ents])

S1: [('comune', 'ADJ', 'ROOT', 'comune'), ('2', 'NUM', 'nummod', '743'), ('743', 'NUM', 'nummod', 'italiano'), ('357', 'NUM', 'nummod', 'abitanti'), ('abitanti', 'NOUN', 'nsubj', 'comune')]
S2: [('fondata', 'fondare', 'acl'), ('secolo', 'secolo', 'obl:agent'), ('suggeriscano', 'suggeriscare', 'advcl')] [('Napoli', 'LOC'), ('Partenope', 'LOC'), ('Cumani', 'MISC'), ('Pizzofalcone', 'LOC')]
S3: [('abitato', 'NOUN', 'nsubj:pass'), ('venne', 'AUX', 'aux:pass'), ('rifondato', 'VERB', 'ROOT'), ('venne', 'AUX', 'aux:pass'), ('Grande', 'PROPN', 'nsubj:pass')] [('Romani', 'LOC'), ('I secolo a.C.', 'MISC'), ('Impero', 'LOC'), ('Teodorico il Grande', 'PER')]


| # | Sentence | Correct ✔ | Wrong ✘ |
|---|---|---|---|
| 1 | *Roma è un comune italiano di 2 743 357 abitanti, capoluogo dell'omonima città metropolitana e della regione Lazio, nonché capitale della Repubblica Italiana.* | *Roma*, *Lazio* and *Repubblica Italiana* are found as entities; the contraction *dell'* is split off and lemmatised as *di il*. | *comune* is tagged **ADJ** instead of NOUN, and because the number is split into three tokens, *abitanti* becomes a second **nsubj** instead of a modifier of *comune*. |
| 2 | *Napoli nasce come Partenope, fondata dai Cumani su Pizzofalcone a fine VIII secolo a.C. su base archeologica, sebbene sporadici reperti suggeriscano una datazione più antica.* | *Napoli*, *Partenope* (the old Greek name) and the hill *Pizzofalcone* are all found as **LOC**; ROOT is *nasce*, and *fondata → fondare* is correctly attached to *Partenope*. | The subjunctive *suggeriscano* gets the non-existing lemma ***suggeriscare*** (should be *suggerire*); *secolo* is analysed as **obl:agent** (the agent of a passive), but it is a time expression. |
| 3 | *Di origine preistorica, l'abitato venne rifondato dai Romani all'interno dell'ansa del fiume …, quando venne occupato dal re germanico Teodorico il Grande.* | The passive with *venire* is analysed correctly: *abitato* = **nsubj:pass**, *venne* = **aux:pass**, ROOT = *rifondato*; *Teodorico il Grande* = **PER**. | *Romani* (a people) and *Impero* are labelled **LOC**; *Grande*, part of the name, becomes a separate subject (**nsubj:pass**) of *occupato*. |

Additional observation: the Italian model only has the labels **PER, LOC, ORG, MISC** – there is
no DATE label, so *I secolo a.C.* is labelled MISC and many dates are not found at all.

### 5.2 Long words, accents and contractions

In [14]:
for s in ["l'uomo dell'Italia, del nel all'inizio", "Kraftfahrzeug-Haftpflichtversicherung",
          "L'Università è più antica dell'Europa", "c'è un po' di caffè"]:
    d = nlp(s)
    print(s)
    print('   tokens:', [t.text for t in d])
    print('   lemmas:', [t.lemma_ for t in d])

l'uomo dell'Italia, del nel all'inizio
   tokens: ["l'", 'uomo', "dell'", 'Italia', ',', 'del', 'nel', "all'", 'inizio']
   lemmas: ['il', 'uomo', 'di il', 'Italia', ',', 'di il', 'in il', 'a il', 'inizio']
Kraftfahrzeug-Haftpflichtversicherung
   tokens: ['Kraftfahrzeug-Haftpflichtversicherung']
   lemmas: ['Kraftfahrzeug-Haftpflichtversicherung']
L'Università è più antica dell'Europa
   tokens: ["L'", 'Università', 'è', 'più', 'antica', "dell'", 'Europa']
   lemmas: ['il', 'università', 'essere', 'più', 'antico', 'di il', 'Europa']
c'è un po' di caffè
   tokens: ["c'", 'è', 'un', "po'", 'di', 'caffè']
   lemmas: ['ci', 'essere', 'uno', 'poco', 'di', 'caffè']


* Elided articles and prepositions (*l'*, *dell'*, *all'*, *c'*) are separate tokens and are
  lemmatised to *il*, *di il*, *a il* – good for counting words.
* Articulated prepositions without an apostrophe (*del*, *nel*) stay **one token** but receive the
  two-word lemma *di il* / *in il*.
* Accented letters (*è, più, Università, caffè*) are handled correctly because the corpus is UTF-8.
* A long German compound is not split – the Italian tokenizer only splits on spaces and punctuation.

## 6. Our row of the comparison table (Section 6.6)

In [15]:
with open('sources.csv', encoding='utf-8') as f:
    sources = list(csv.DictReader(f))

row = {
    'Language': 'Italian',
    'Source': 'it.wikipedia.org (Italian cities, opening paragraphs)',
    'Licence': 'CC BY-SA 4.0',
    'Texts': len(texts),
    'Tokens': len(tokens),
    'Unique words': len(set(tokens)),
    'Problems found': 'numbers split by spaces; 1 template leftover; IPA brackets; same opening pattern in all texts',
}
for k, v in row.items():
    print(f'{k:<15}{v}')

Language       Italian
Source         it.wikipedia.org (Italian cities, opening paragraphs)
Licence        CC BY-SA 4.0
Texts          20
Tokens         5389
Unique words   1646
Problems found numbers split by spaces; 1 template leftover; IPA brackets; same opening pattern in all texts


| Language | Source | Licence | Texts | Tokens | Unique words | Problems found |
|---|---|---|---|---|---|---|
| Italian | it.wikipedia.org (20 Italian cities) | CC BY-SA 4.0 | 20 | 5389 | 1646 | numbers split by spaces, 1 template leftover (*Gamma -.*), IPA pronunciation brackets, same opening pattern, ADJ/NOUN and lemma errors, no DATE entities |

**Discussion**

1. *Most problems:* for Italian they come from formatting (numbers with spaces, lost templates,
   pronunciation brackets) and from ambiguous or rare words (*comune* noun vs. adjective,
   unusual verb forms such as *suggeriscano* that get a wrong lemma).
2. *Long words / accents / contractions:* see Section 5.2 – elisions with apostrophe are split,
   *del/nel* are kept as one token with a two-word lemma, accents are fine in UTF-8.
3. *What we would change:* normalise numbers (remove the space separators), remove template
   leftovers, use more varied parts of the articles (history, economy, culture) so that not every
   text starts with the same sentence, grow the corpus to ~100 cities/texts, and keep
   one sentence per line to make later tokenization and n-gram work easier.

## 7. Sources and licences

In [16]:
for s in sources:
    print(f"{s['file']}  {s['title']:<25} {s['url']}  ({s['date_accessed']}, {s['licence']})")

text01.txt  Roma                      https://it.wikipedia.org/wiki/Roma  (2026-11-15, CC BY-SA 4.0)
text02.txt  Milano                    https://it.wikipedia.org/wiki/Milano  (2026-11-15, CC BY-SA 4.0)
text03.txt  Napoli                    https://it.wikipedia.org/wiki/Napoli  (2026-11-15, CC BY-SA 4.0)
text04.txt  Torino                    https://it.wikipedia.org/wiki/Torino  (2026-11-15, CC BY-SA 4.0)
text05.txt  Palermo                   https://it.wikipedia.org/wiki/Palermo  (2026-11-15, CC BY-SA 4.0)
text06.txt  Genova                    https://it.wikipedia.org/wiki/Genova  (2026-11-15, CC BY-SA 4.0)
text07.txt  Bologna                   https://it.wikipedia.org/wiki/Bologna  (2026-11-15, CC BY-SA 4.0)
text08.txt  Firenze                   https://it.wikipedia.org/wiki/Firenze  (2026-11-15, CC BY-SA 4.0)
text09.txt  Bari                      https://it.wikipedia.org/wiki/Bari  (2026-11-15, CC BY-SA 4.0)
text10.txt  Catania                   https://it.wikipedia.org/wiki/Catani

## 8. Preprocessing pipeline (`preprocess.py`)

Steps: **cleaning** (apostrophes, template leftovers) → **normalisation** (numbers written with spaces
are joined, `2 743 357` → `2743357`) → **sentence splitting + tokenization** → **POS / lemma / NER** →
**filtering** (lowercase, only alphabetic tokens, no stop words) → content lemmas.

Output: `corpus_clean/` (one clean sentence per line), `corpus_processed/` (lemmas per sentence),
`output/preprocessing_stats.json`, `output/search_index.json`.

In [17]:
import json, preprocess
preprocess.main()

text01.txt: Roma             11 sentences
text02.txt: Milano           12 sentences
text03.txt: Napoli           12 sentences


text04.txt: Torino           8 sentences
text05.txt: Palermo          11 sentences


text06.txt: Genova           12 sentences
text07.txt: Bologna          13 sentences
text08.txt: Firenze          10 sentences
text09.txt: Bari             15 sentences


text10.txt: Catania          10 sentences
text11.txt: Venezia          13 sentences
text12.txt: Verona           11 sentences
text13.txt: Messina          13 sentences


text14.txt: Padova           12 sentences
text15.txt: Trieste          12 sentences
text16.txt: Brescia          12 sentences


text17.txt: Parma            14 sentences
text18.txt: Taranto          9 sentences


text19.txt: Modena           15 sentences
text20.txt: Reggio Calabria  11 sentences

236 sentences | tokens 6618 -> alpha 5389 -> content 2817
unique forms 1646 -> unique lemmas 1290 | fixes {'numbers joined': 25, 'template leftovers removed': 1}


In [18]:
from preprocess import clean_text
before = texts[0].split('.')[0] + '.'
after, fixes = clean_text(before)
print('BEFORE:', [t.text for t in nlp(before)][:10])
print('AFTER :', [t.text for t in nlp(after)][:8])
print()
print('clean    :', Path('corpus_clean/text01.txt').read_text(encoding='utf-8').splitlines()[0])
print('processed:', Path('corpus_processed/text01.txt').read_text(encoding='utf-8').splitlines()[0])

BEFORE: ['Roma', 'è', 'un', 'comune', 'italiano', 'di', '2', '743', '357', 'abitanti']


AFTER : ['Roma', 'è', 'un', 'comune', 'italiano', 'di', '2743357', 'abitanti']

clean    : Roma è un comune italiano di 2743357 abitanti, capoluogo dell'omonima città metropolitana e della regione Lazio, nonché capitale della Repubblica Italiana.
processed: roma comune italiano abitante capoluogo omonimo metropolitano regione lazio nonché capitale repubblica italiano


In [19]:
st = json.load(open('output/preprocessing_stats.json', encoding='utf-8'))
for k in ['texts', 'sentences', 'tokens_all', 'tokens_alpha', 'tokens_content', 'unique_forms', 'unique_lemmas', 'fixes']:
    print(f'{k:<16}{st[k]}')
print('città is a spaCy stop word:', 'città' in nlp.Defaults.stop_words)

texts           20
sentences       236
tokens_all      6618
tokens_alpha    5389
tokens_content  2817
unique_forms    1646
unique_lemmas   1290
fixes           {'numbers joined': 25, 'template leftovers removed': 1}
città is a spaCy stop word: True


**Effect:** removing stop words drops about half of the words; lemmatisation reduces the vocabulary
from 1646 forms to 1290 lemmas. Note: spaCy's Italian stop list contains *città*, a content word in our
corpus, so in Phase 2 we will use a custom stop list.

## 9. Application area: a search engine for Italian cities

The user writes a question in Italian; the query goes through the **same preprocessing** (lowercase,
lemma, stop words removed); every corpus sentence is a **TF-IDF** vector of lemmas, and sentences are ranked
by **cosine similarity**. The same algorithm runs in the browser in `index.html` (web presentation).

In [20]:
import math
from collections import Counter

ix = json.load(open('output/search_index.json', encoding='utf-8'))
sents = ix['sentences']
N = len(sents)
df = Counter(l for s in sents for l in set(s['lemmas']))
idf = lambda l: math.log((N + 1) / (df[l] + 1)) + 1

def vec(lemmas):
    tf = Counter(lemmas)
    v = {l: c * idf(l) for l, c in tf.items()}
    return v, math.sqrt(sum(x * x for x in v.values())) or 1

S = [vec(s['lemmas']) for s in sents]
STOP = set(ix['stop'])
F2L = ix['form2lemma']

def search(query, k=3):
    # same as the web demo: lowercase, form -> lemma from the corpus table, stop words removed
    words = re.findall(r"[a-zà-öø-ÿ]+", query.lower())
    q = [F2L.get(w, w) for w in words if w not in STOP and F2L.get(w, w) not in STOP]
    qv, qn = vec(q)
    scores = sorted(((sum(w * sv.get(l, 0) for l, w in qv.items()) / (qn * sn), i) for i, (sv, sn) in enumerate(S)), reverse=True)
    print('Query:', query, '->', q)
    for sc, i in scores[:k]:
        if sc > 0:
            print(f"  {sc:.3f}  {ix['docs'][sents[i]['doc']]['title']:<10} {sents[i]['text'][:100]}")

for q in ['università più antica', 'porto sul mare', 'patrimonio UNESCO', 'fondata dai Greci']:
    search(q)

Query: università più antica -> ['università', 'antico']
  0.682  Padova     Dal 1222 Padova è sede di un'università tra le più antiche del mondo, la seconda in Italia.
  0.573  Bologna    Importante centro universitario, culturale ed economico, è sede dell'Università di Bologna, tradizio
  0.348  Catania    Sotto la dinastia aragonese fu capitale del Regno di Sicilia, e dal 1434 per volere del re Alfonso V
Query: porto sul mare -> ['porto', 'mare']
  0.510  Trieste    Il porto di Trieste dal 2013 è il porto italiano con il maggior traffico merci ed è uno dei più impo
  0.347  Genova     Il principale simbolo monumentale cittadino è la Lanterna, storico faro del porto.
La città è situat
  0.272  Bari       La città va dai 0 ai 131 metri di altitudine sul livello del mare.
Query: patrimonio UNESCO -> ['patrimonio', 'unesco']
  0.533  Verona     Verona è stata dichiarata patrimonio dell'umanità dall'UNESCO per le peculiarità urbanistiche e per 
  0.458  Padova     A Padova sono presenti 

**Next (Phase 2):** a larger corpus (~100 cities, more sections), word vectors / sentence embeddings,
entity-aware ranking, a chatbot interface and an evaluation with test questions (precision@k).